In [0]:
--Validate mandatory member fields
SELECT
    COUNT(*) AS Total_Records,

    SUM(CASE
        WHEN Member_Name IS NULL OR TRIM(Member_Name) = ''
        THEN 1 ELSE 0
    END) AS Missing_Member_Name,

    SUM(CASE
        WHEN Enrollment_Date IS NULL
        THEN 1 ELSE 0
    END) AS Invalid_Enrollment_Date,

    SUM(CASE
        WHEN Member_ID IS NULL OR TRIM(CAST(Member_ID AS STRING)) = ''
        THEN 1 ELSE 0
    END) AS Missing_Member_ID

FROM delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/`;

Total_Records,Missing_Member_Name,Invalid_Enrollment_Date,Missing_Member_ID
9,0,1,0


In [0]:
--Invalid records
SELECT
    Member_Name,
    Member_ID,
    Source_Country,
    Enrollment_Date,
    Enrollment_Date_Status
FROM delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/`
WHERE Enrollment_Date_Status <> 'VALID'
   OR Enrollment_Date IS NULL;

Member_Name,Member_ID,Source_Country,Enrollment_Date,Enrollment_Date_Status
Jonnathan,2,AUS,null,INVALID


In [0]:
--Duplicate members record
SELECT
    Member_Name,
    COUNT(*) AS Record_Count
FROM delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/`
GROUP BY Member_Name
HAVING COUNT(*) > 1;

Member_Name,Record_Count
Mike,2


In [0]:
--Identify duplicate record

SELECT
    Member_Name,
    Member_ID,
    Source_Country,
    Enrollment_Date,
    Last_Flight_Date
FROM delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/`
WHERE Member_Name = 'Mike'
ORDER BY Enrollment_Date DESC;

Member_Name,Member_ID,Source_Country,Enrollment_Date,Last_Flight_Date
Mike,1,AUS,2022-05-11,2022-08-01
Mike,3,USA,2021-12-28,2021-12-30


In [0]:
--Validate DOB quality

SELECT
    Member_Name,
    Member_ID,
    Source_Country,
    DOB,
    DOB_Status
FROM delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/`
WHERE DOB_Status <> 'VALID'
   OR DOB IS NULL;

Member_Name,Member_ID,Source_Country,DOB,DOB_Status
Mike,1,AUS,null,INVALID
Sam,1,USA,null,MISSING
John,2,USA,null,MISSING
Mike,3,USA,null,MISSING


In [0]:
--Validate redemption transaction
SELECT
    COUNT(*) AS Total_Transactions,

    SUM(CASE
        WHEN Transaction_ID IS NULL OR TRIM(Transaction_ID) = ''
        THEN 1 ELSE 0
    END) AS Missing_Transaction_ID,

    SUM(CASE
        WHEN Miles_Redeemed IS NULL OR Miles_Redeemed <= 0
        THEN 1 ELSE 0
    END) AS Invalid_Miles,

    SUM(CASE
        WHEN Transaction_Status NOT IN ('COMPLETED', 'PENDING')
        THEN 1 ELSE 0
    END) AS Invalid_Status

FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/redemption_transactions/`;

Total_Transactions,Missing_Transaction_ID,Invalid_Miles,Invalid_Status
2,0,0,0


In [0]:
--Duplicate Transaction

SELECT
    Transaction_ID,
    COUNT(*) AS Record_Count
FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/redemption_transactions/`
GROUP BY Transaction_ID
HAVING COUNT(*) > 1;

Transaction_ID,Record_Count


In [0]:
--Create a consolidated data quality summary

SELECT 'Member Records' AS Data_Category,
       COUNT(*) AS Total_Records,
       SUM(CASE WHEN Member_Name IS NULL OR TRIM(Member_Name) = ''
                THEN 1 ELSE 0 END) AS Missing_Required_Fields,
       SUM(CASE WHEN Enrollment_Date_Status = 'INVALID'
                THEN 1 ELSE 0 END) AS Invalid_Records
FROM delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/`

UNION ALL

SELECT 'Redemption Transactions',
       COUNT(*),
       SUM(CASE WHEN Transaction_ID IS NULL OR TRIM(Transaction_ID) = ''
                THEN 1 ELSE 0 END),
       SUM(CASE WHEN Miles_Redeemed IS NULL OR Miles_Redeemed <= 0
                     OR Transaction_Status NOT IN ('COMPLETED', 'PENDING')
                THEN 1 ELSE 0 END)
FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/redemption_transactions/`;

Data_Category,Total_Records,Missing_Required_Fields,Invalid_Records
Member Records,9,0,1
Redemption Transactions,2,0,0


In [0]:
--Save data quality summary

CREATE OR REPLACE TABLE delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/data_quality_summary/`
USING DELTA
AS

SELECT 'Member Records' AS Data_Category,
       COUNT(*) AS Total_Records,
       SUM(CASE WHEN Member_Name IS NULL OR TRIM(Member_Name) = ''
                THEN 1 ELSE 0 END) AS Missing_Required_Fields,
       SUM(CASE WHEN Enrollment_Date_Status = 'INVALID'
                THEN 1 ELSE 0 END) AS Invalid_Records
FROM delta.`abfss://silver@incubytedatalake01.dfs.core.windows.net/member_data/`

UNION ALL

SELECT 'Redemption Transactions',
       COUNT(*),
       SUM(CASE WHEN Transaction_ID IS NULL OR TRIM(Transaction_ID) = ''
                THEN 1 ELSE 0 END),
       SUM(CASE WHEN Miles_Redeemed IS NULL OR Miles_Redeemed <= 0
                     OR Transaction_Status NOT IN ('COMPLETED', 'PENDING')
                THEN 1 ELSE 0 END)
FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/redemption_transactions/`;

num_affected_rows,num_inserted_rows


In [0]:
SELECT *
FROM delta.`abfss://gold@incubytedatalake01.dfs.core.windows.net/data_quality_summary/`;

Data_Category,Total_Records,Missing_Required_Fields,Invalid_Records
Redemption Transactions,2,0,0
Member Records,9,0,1
